# Stability of explicit vs implicit methods

Compare **forward Euler**, **backward Euler** and **RK45** to see how the time-step size affects numerical stability.

## 1. Problem statement

We will use the reaction $A \overset{k}{\rightarrow} B$:

$$\frac{dC_A}{dt} = -kC_A, \qquad C_A(0)=1\ \mathrm{mol/L}, \qquad k=0.2\ \mathrm{min}^{-1}.$$

Assuming a constant total concentration of $1\ \mathrm{mol/L}$, $C_B = 1 - C_A$, $C_B(0)=0$.

The exact solutions are $C_A(t)=C_A(0)e^{-kt}$ and $C_B(t)=1-C_A(0)e^{-kt}$.

In [ ]:
# Import libraries
from collections.abc import Callable
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.integrate import solve_ivp

# Problem parameters
k = 0.2  # 1/min
c0 = np.array([1.0])  # mol/L

## 2. Reuse the forward and backward Euler methods

Reuse the `forward_euler` and `backward_euler` functions defined in the previous examples.

In [ ]:
def forward_euler(func: Callable, c0: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Generic forward Euler method for initial value problem.

    Parameters
    ----------
    func : Callable
        ODE system to be solved
    c0 : np.ndarray
        Initial condition
    t : np.ndarray
        Time grid points

    Returns
    -------
    np.ndarray
        Solution of ODE system
    """
    # initialize arrays for time and solution values
    c = np.zeros((len(c0), len(t)))
    h = t[1] - t[0]

    # initial condition
    c[:, 0] = c0

    # iterate over each time step
    for i in range(1, len(t)):
        c[:, i] = c[:, i - 1] + h * func(t[i - 1], c[:, i - 1])
    return c

In [ ]:
def backward_euler(func: Callable, c0: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Generic backward Euler method for initial value problem. Use scipy's
    fsolve to solve root finding problem.

    Parameters
    ----------
    func : Callable
        Function that defines the ODE (y' = func(t, y)).
    c0 : np.ndarray
        Initial condition.
    t : np.ndarray
        Time domain.

    Returns
    -------
    np.ndarray
        Array of solution values at the time points.
    """
    # initialize arrays for time and solution values
    c = np.zeros([len(c0), len(t)])
    dt = t[1] - t[0]

    # initial condition
    c[:, 0] = c0

    # iterate over each time step
    for i in range(len(t) - 1):
        # initial guess for y_{i+1}
        c_guess = c[:, i]

        # define backward Euler function
        def euler(c_next: np.ndarray, i: int = i) -> np.ndarray:
            return c[:, i] + dt * func(t[i + 1], c_next) - c_next

        # update solution
        c[:, i + 1] = scipy.optimize.fsolve(euler, c_guess)

    return c

## 3. Define the ODE and the analytical solution

In [ ]:
# ODE definition
def dcA(t, c):
    return -k * c


# Analytical solution
def analytical_c(t):
    return c0[0] * np.exp(-k * t)

## 4. Compare stability on a fine and a coarse grid

The forward Euler method becomes unstable once the step size is too large relative to $k$, the backward Euler method and RK45 remain stable regardless.

In [ ]:
# Fine grid: forward Euler is stable
t_fine = np.linspace(0, 50, 51)
h_fine = t_fine[1] - t_fine[0]
c_fine = forward_euler(dcA, c0, t_fine)

# Coarse grid: forward Euler becomes unstable, backward Euler and RK45 stays stable
t_coarse = np.linspace(0, 50, 5)
h_coarse = t_coarse[1] - t_coarse[0]
c_coarse_forward = forward_euler(dcA, c0, t_coarse)
c_coarse_backward = backward_euler(dcA, c0, t_coarse)
c_coarse_solveivp = solve_ivp(fun=dcA, t_span=(t_coarse[0], t_coarse[-1]), y0=c0, t_eval=t_coarse, method="RK45").y[0]

## 5. Plot results

In [ ]:
fig, ax = plt.subplots()
ax.plot(t_fine, c_fine[0, :], label=f"Forward Euler, h={h_fine:.0f} min")
ax.plot(t_coarse, c_coarse_forward[0, :], "o-", label=f"Forward Euler, h={h_coarse:.0f} min")
ax.plot(t_coarse, c_coarse_backward[0, :], "s-", label=f"Backward Euler, h={h_coarse:.0f} min")
ax.plot(t_coarse, c_coarse_solveivp, "d-", label=f"RK45, h={h_coarse:.0f} min")
ax.plot(t_fine, analytical_c(t_fine), "k--", label="Analytical solution")
ax.set(xlabel="Time [min]", ylabel="Concentration [mol/L]",
       title="Stability analysis")
ax.legend()
ax.grid(alpha=0.3)
plt.show()